# Limpieza de Datos v3 · Calidad y Preprocesamiento
**Equipo:** Calidad de Datos (Carlo Kiliano Ferrera, José Julian Pérez, Aldo Sebastián Altamirano)
**Curso:** Calidad y Preprocesamiento de Datos · Licenciatura en Ciencia de Datos · IIMAS UNAM CU · 2026‑2
**Framework:** DAMA‑DMBOK · Fase *Improve*

---



## Cambios respecto a v2

| # | Cambio | Motivo |
|---|---|---|
| 1 | **Sección 4.5 ampliada**: SACMEX concatena 2018-2021 (`reportes_agua_hist.csv`) + 2022-2024 | Serie temporal completa para XGBoost del README |
| 2 | **Sección 4.8 NUEVA**: limpieza del shapefile IECM 2022 | Genera `colonias_iecm_limpio.csv` y `colonias_iecm.geojson` con polígonos validados, normalización de nombres y mapeo CDT→cve_alcaldia |

`Fusion_v6` consumirá estos dos artefactos y migrará el universo territorial de SEPOMEX a IECM.

## Objetivo del notebook

Ejecutar las decisiones de limpieza derivadas de `Perfilado_v2.ipynb` y entregar **siete CSVs limpios y filtrados a CDMX** listos para `fusion.ipynb`. Cada decisión que se aplica aquí está justificada por una métrica medida en el perfilado; nada es discrecional.

## Reglas que sigue este notebook (separación de fases DAMA)

1. **Cada operación tiene un "antes/después"**. Toda celda de limpieza imprime *n_filas_in*, *n_filas_out* y la diferencia, para auditoría.
2. **Cero invención de datos**. Las filas con coordenadas inválidas se *marcan*, no se inventan; los nulos sólo se imputan cuando la columna tiene < 30 % de nulos y la imputación es trazable (mediana por grupo, KNN, etc.).
3. **Trazabilidad por columna**. Toda transformación se documenta en un log que se exporta junto con los datos.
4. **Idempotencia**. Re‑ejecutar el notebook produce exactamente los mismos archivos de salida.
5. **Catálogo canónico de alcaldías**. Las 16 alcaldías CDMX se referencian con su clave INEGI (002‑017). Cada dataset limpio incluye `cve_alcaldia` para joins triviales en `fusion.ipynb`.

## Hallazgos del perfilado que dirigen este notebook

| Fuente | Decisión que se aplica aquí |
|---|---|
| **Morbilidad SSA** | Filtro CDMX × 8 enfermedades hídricas (8 filas). La regla `ACUMULADO == TOTALM + TOTALF` se cumple al 100 %; no requiere reparación |
| **INEGI ITER** | Coordenadas en formato DMS → decimal. Localidades confidenciales (LOC=9998/9999) y totales (LOC=0) se eliminan. Localidades pequeñas con datos protegidos por confidencialidad se eliminan |
| **CONEVAL** | Reemplazar `-999` por `NaN`. *DROP* columna `carencia_servicios_basicos_vivienda_porcentaje` (80 % nulos). *IMPUTE* `carencia_calidad_espacios_vivienda_porcentaje` (19 %) con mediana por alcaldía‑año |
| **SEPOMEX** | Reparar formato de CP (956 con formato inválido por padding fallido). Validar rango CDMX 01000‑16999 |
| **SACMEX** | Parseo robusto de fechas (60 % no parseables con `dayfirst=True`). Eliminar lag negativo (14 654), duplicados de fila completa (12 729). Mantener duplicados de folio (son updates legítimos) |
| **CONAGUA Sitios** | Filtrar a territorio mexicano (15 sitios fuera). Conservar nacional: los abastecedores de CDMX (Cutzamala, Lerma) están en EdoMex/Michoacán |
| **CONAGUA Resultados** | Aplicar tabla de decisión: *DROP* 331 columnas (>95 % nulos), *FLAG* 103, *IMPUTE* 27 vía KNN, *KEEP* 11. Parsear `<`, `>`, `ND` en 71 columnas. Filtrar regla DBO ≤ DQO |

## Estructura del notebook

0. Configuración del entorno y constantes
1. Funciones reutilizables de limpieza
2. Catálogo canónico de alcaldías CDMX
3. Limpieza por fuente (una sección, un CSV de salida):
   * 3.1 Morbilidad SSA → `morbilidad_cdmx_limpio.csv`
   * 3.2 INEGI ITER → `inegi_cdmx_limpio.csv`
   * 3.3 CONEVAL → `coneval_cdmx_limpio.csv`
   * 3.4 SEPOMEX → `sepomex_cdmx_limpio.csv`
   * 3.5 SACMEX → `sacmex_cdmx_limpio.csv`
   * 3.6 CONAGUA Sitios → `conagua_sitios_limpio.csv`
   * 3.7 CONAGUA Resultados → `conagua_resultados_cdmx_limpio.csv`
4. Verificación post‑limpieza
5. Catálogo de salidas y log de transformaciones
6. Conclusiones

## 1. Configuración del entorno

Imports, raíz del proyecto (idéntica al perfilado), rutas de entrada y de salida. Si la carpeta `datos/datos_limpios/` no existe se crea.

In [2]:
# Instalación de dependencias (descomentar si es la primera vez)
# !pip install pandas numpy scikit-learn pyyaml

In [3]:
import os
import re
import json
import warnings
import unicodedata
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

### 1.1 Raíz del proyecto y rutas

In [4]:
def get_project_root(marker: str = "README.md") -> Path:
    """Sube el árbol de directorios hasta hallar el archivo `marker`."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del proyecto (no existe README.md hacia arriba).")

PROJECT_ROOT  = get_project_root()
DATOS_CRUDOS  = PROJECT_ROOT / "datos" / "datos_crudos"
DATOS_LIMPIOS = PROJECT_ROOT / "datos" / "datos_limpios"
DATOS_LIMPIOS.mkdir(parents=True, exist_ok=True)

print(f"📂 PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"📂 DATOS_CRUDOS : {DATOS_CRUDOS}")
print(f"📂 DATOS_LIMPIOS: {DATOS_LIMPIOS}")
assert DATOS_CRUDOS.exists()

📂 PROJECT_ROOT  : c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos
📂 DATOS_CRUDOS : c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos\datos\datos_crudos
📂 DATOS_LIMPIOS: c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos\datos\datos_limpios


In [5]:
# Rutas de entrada (idénticas al perfilado)
RUTAS_IN = {
    "morbilidad_ssa":  DATOS_CRUDOS / "Anuario_Morbilidad_2017.csv",
    "inegi_iter":      DATOS_CRUDOS / "conjunto_de_datos_iter_00CSV20.csv",
    "coneval_pobreza": DATOS_CRUDOS / "pobreza_grupos_poblacionales_edad.csv",
    "sepomex":         DATOS_CRUDOS / "catalogo_sepomex.csv",
    "sacmex_reportes": DATOS_CRUDOS / "reportes_agua_2024_01.csv",
    "conagua_sitios":  DATOS_CRUDOS / "Sitios_CONAGUA.csv",
    "conagua_result":  DATOS_CRUDOS / "Resultados_CONAGUA.csv",
    "sacmex_hist":     DATOS_CRUDOS / "reportes_agua_hist.csv",  # NUEVO v3
    "iecm_shp":        DATOS_CRUDOS / "colonias_iecm2022_.shp",     # NUEVO v3

}

# Rutas de salida
RUTAS_OUT = {
    "morbilidad":           DATOS_LIMPIOS / "morbilidad_cdmx_limpio.csv",
    "inegi":                DATOS_LIMPIOS / "inegi_cdmx_limpio.csv",
    "coneval":              DATOS_LIMPIOS / "coneval_cdmx_limpio.csv",
    "sepomex":              DATOS_LIMPIOS / "sepomex_cdmx_limpio.csv",
    "sacmex":               DATOS_LIMPIOS / "sacmex_cdmx_limpio.csv",
    "conagua_sitios":       DATOS_LIMPIOS / "conagua_sitios_limpio.csv",
    "conagua_resultados":   DATOS_LIMPIOS / "conagua_resultados_cdmx_limpio.csv",
    "catalogo_alcaldias":   DATOS_LIMPIOS / "catalogo_alcaldias_cdmx.csv",
    "iecm":                 DATOS_LIMPIOS / "colonias_iecm_limpio.csv",       # NUEVO v3
    "iecm_geojson":         DATOS_LIMPIOS / "colonias_iecm.geojson",           # NUEVO v3
    "log_limpieza":         DATOS_LIMPIOS / "_log_limpieza.json",
}

for k, p in RUTAS_IN.items():
    flag = "✅" if p.exists() else "⚠️"
    print(f"  {flag}  {k:<18} -> {p.name}")

  ✅  morbilidad_ssa     -> Anuario_Morbilidad_2017.csv
  ✅  inegi_iter         -> conjunto_de_datos_iter_00CSV20.csv
  ✅  coneval_pobreza    -> pobreza_grupos_poblacionales_edad.csv
  ✅  sepomex            -> catalogo_sepomex.csv
  ✅  sacmex_reportes    -> reportes_agua_2024_01.csv
  ✅  conagua_sitios     -> Sitios_CONAGUA.csv
  ✅  conagua_result     -> Resultados_CONAGUA.csv
  ✅  sacmex_hist        -> reportes_agua_hist.csv
  ✅  iecm_shp           -> colonias_iecm2022_.shp


### 1.2 Constantes globales

Recuperamos las mismas constantes que el perfilado (bounding box CDMX, NOM‑127, códigos de missing) y agregamos las nuevas que necesita la limpieza: umbrales de imputación, columnas a conservar por fuente, mapeo de prefijos químicos.

In [6]:
# Bounding box CDMX (Marco Geoestadístico INEGI 2020)
CDMX_BBOX = {"lat_min": 19.05, "lat_max": 19.59, "lon_min": -99.36, "lon_max": -98.94}

# Bounding box México (validez para CONAGUA Sitios nacional)
MX_BBOX = {"lat_min": 14.5, "lat_max": 32.7, "lon_min": -118.5, "lon_max": -86.5}

# Códigos de missing por fuente (centralizados desde el perfilado)
MISSING_CODES = {
    "INEGI":   ["*", "N/D", "N/A"],
    "CONEVAL": [-999, -999.0, "-999"],
}

# NOM-127-SSA1-2021 (los mismos límites del perfilado)
NOM_127 = {
    "AS_TOT":        {"limite": 0.010, "unidad": "mg/L",      "param": "Arsénico"},
    "PB_TOT":        {"limite": 0.010, "unidad": "mg/L",      "param": "Plomo"},
    "HG_TOT":        {"limite": 0.006, "unidad": "mg/L",      "param": "Mercurio"},
    "CD_TOT":        {"limite": 0.005, "unidad": "mg/L",      "param": "Cadmio"},
    "FLUORUROS_TOT": {"limite": 1.500, "unidad": "mg/L",      "param": "Fluoruros"},
    "N_NO3":         {"limite": 10.00, "unidad": "mg/L",      "param": "Nitratos"},
    "E_COLI":        {"limite": 0.000, "unidad": "NMP/100mL", "param": "E. coli"},
    "COLI_FEC":      {"limite": 0.000, "unidad": "NMP/100mL", "param": "Coliformes fecales"},
}

# Umbrales para tabla de decisión (mismos del perfilado)
DECISION_THRESHOLDS = {"DROP": 95.0, "FLAG": 30.0, "IMPUTE": 5.0}

# Diagnósticos hídricos confirmados en el perfilado de Morbilidad
ENFERMEDADES_HIDRICAS = [
    "Cólera",
    "Amebiasis intestinal",
    "Shigelosis",
    "Fiebre tifoidea",
    "Fiebre paratifoidea y otras salmonelosis",
    "Fiebre paratifoidea",
    "Giardiasis",
    "Otras infecciones intestinales debidas a protozoarios",
    "Infecciones intestinales por otros organismos y las mal definidas",
]

# Log de transformaciones aplicadas (se exporta como JSON al final)
LOG = defaultdict(dict)

## 2. Funciones reutilizables

Las cinco operaciones que se repiten en varias fuentes se factoriza aquí. La función más delicada es `parsear_coords_dms`: el ITER del INEGI publica las coordenadas en formato grados‑minutos‑segundos como `19°25'42.394"`, lo cual explica el 100 % de "nulos" reportado por el perfilado tras la coerción a numérico.

In [7]:
def normalizar_texto(serie: pd.Series) -> pd.Series:
    """NFKD + sin acentos + mayúsculas + sin espacios laterales."""
    return (serie
            .fillna("")
            .astype(str)
            .str.normalize("NFKD")
            .str.encode("ascii", errors="ignore")
            .str.decode("utf-8")
            .str.upper()
            .str.strip())


def parsear_coords_dms(serie: pd.Series) -> pd.Series:
    """Convierte coordenadas en formato DMS (19°25'42.394\") a decimales.

    Estrategia en cascada:
      1) intenta numérico directo (por si ya viene en decimal),
      2) reemplaza coma decimal por punto y reintenta,
      3) parsea formato DMS con regex,
      4) marca como NaN si nada funcionó.
    """
    if serie.dtype.kind in "fi":
        return serie.astype(float)

    s = serie.astype(str).str.strip()
    out = pd.to_numeric(s, errors="coerce")
    pendientes = out.isna() & s.notna() & (s != "") & (s.str.lower() != "nan")

    # Intento 2: coma decimal
    if pendientes.any():
        s_pt = s[pendientes].str.replace(",", ".", regex=False)
        out.loc[pendientes] = pd.to_numeric(s_pt, errors="coerce")
        pendientes = out.isna() & s.notna() & (s != "") & (s.str.lower() != "nan")

    # Intento 3: regex DMS  (capta °, ', " con o sin espacios)
    if pendientes.any():
        regex_dms = re.compile(
            r"^\s*(-?\d+(?:\.\d+)?)[°ºd:\s]+(\d+(?:\.\d+)?)[\'m:\s]+(\d+(?:\.\d+)?)[\"s]?\s*([NSEWnsew]?)\s*$"
        )
        for idx in s.index[pendientes]:
            m = regex_dms.match(s[idx])
            if not m:
                continue
            grados, minutos, segundos, hemis = m.groups()
            valor = abs(float(grados)) + float(minutos)/60 + float(segundos)/3600
            if float(grados) < 0 or hemis.upper() in ("S", "W"):
                valor = -valor
            out.loc[idx] = valor

    return out


def parsear_medicion_quimica(serie: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Parsea mediciones de laboratorio con anotaciones '< X', '> X', 'ND', 'N/D'.

    Convención EPA y mexicana (NMX-AA): valores < límite de detección se imputan
    al medio del límite ( L/2 ). 'ND' / 'N/D' = no detectado → 0 o NaN; preferimos NaN
    porque conservar 0 sesga las medias hacia abajo si la columna ya es escasa.

    Devuelve (serie_numerica, serie_flag) donde flag ∈ {"raw", "below_LOD", "above_LOQ", "ND", "missing"}.
    """
    s = serie.astype(str).str.strip()
    valor = pd.Series(np.nan, index=s.index, dtype="float64")
    flag  = pd.Series("missing", index=s.index, dtype="object")

    mask_nd  = s.str.match(r"^\s*N/?D\s*$", case=False, na=False)
    mask_lt  = s.str.match(r"^\s*<\s*[\d.,]+\s*$", na=False)
    mask_gt  = s.str.match(r"^\s*>\s*[\d.,]+\s*$", na=False)
    mask_num = s.str.match(r"^\s*-?[\d.,]+\s*$", na=False)

    # Numéricos limpios
    valor.loc[mask_num] = pd.to_numeric(s[mask_num].str.replace(",", ".", regex=False), errors="coerce")
    flag.loc[mask_num & valor.notna()] = "raw"

    # < límite de detección → L/2
    sub_lt = s[mask_lt].str.extract(r"<\s*([\d.,]+)")[0].str.replace(",", ".", regex=False)
    valor.loc[mask_lt] = pd.to_numeric(sub_lt, errors="coerce") / 2.0
    flag.loc[mask_lt] = "below_LOD"

    # > límite de cuantificación → conservar valor (cota inferior)
    sub_gt = s[mask_gt].str.extract(r">\s*([\d.,]+)")[0].str.replace(",", ".", regex=False)
    valor.loc[mask_gt] = pd.to_numeric(sub_gt, errors="coerce")
    flag.loc[mask_gt] = "above_LOQ"

    # ND
    flag.loc[mask_nd] = "ND"

    return valor, flag


def parsear_fecha_robusta(serie: pd.Series) -> pd.Series:
    """Parser en cascada para fechas con formatos heterogéneos.

    Justificación: el perfilado halló 60% de fechas no parseables en SACMEX con
    `dayfirst=True`. Causa probable: mezcla de ISO (YYYY-MM-DD) con DMY (DD/MM/YYYY).
    Esta cascada cubre los formatos comunes en datos de gobierno mexicano.
    """
    s = serie.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    formatos = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%d/%m/%Y", "%d/%m/%Y %H:%M:%S",
                "%d-%m-%Y", "%Y/%m/%d", "%m/%d/%Y"]
    for fmt in formatos:
        pendientes = out.isna() & (s != "") & (s.str.lower() != "nan")
        if not pendientes.any():
            break
        intento = pd.to_datetime(s[pendientes], format=fmt, errors="coerce")
        out.loc[pendientes] = intento
    # Último recurso: dejar a pandas inferir
    pendientes = out.isna() & (s != "") & (s.str.lower() != "nan")
    if pendientes.any():
        out.loc[pendientes] = pd.to_datetime(s[pendientes], errors="coerce", dayfirst=True)
    return out


def resumen_paso(nombre_paso: str, n_in: int, n_out: int, detalle: str = ""):
    """Imprime un resumen compacto del paso de limpieza."""
    diff = n_in - n_out
    pct = (diff / max(n_in, 1)) * 100
    flecha = "→"
    print(f"  {nombre_paso:<45}  {n_in:>10,} {flecha} {n_out:>10,}  (-{diff:,} | -{pct:.2f}%)  {detalle}")


def header(title: str, char: str = "="):
    print(f"\n{char*70}\n {title}\n{char*70}")

## 3. Catálogo canónico de alcaldías CDMX

Cada fuente nombra a las alcaldías de manera distinta:
* INEGI: `Coyoacán`, `Cuajimalpa de Morelos`
* CONEVAL: `Coyoacán`, `Cuajimalpa de Morelos`
* SEPOMEX: `COYOACAN`, `CUAJIMALPA DE MORELOS`
* SACMEX: `Coyoacán`, `Cuajimalpa de Morelos`

Para joins triviales en `fusion.ipynb` definimos un catálogo único con la **clave INEGI** (002‑017) como llave canónica. Cualquier nombre normalizado se mapea aquí.

In [8]:
# Catálogo oficial INEGI (claves municipales para entidad 09 = CDMX)
ALCALDIAS_CDMX = pd.DataFrame([
    {"cve_alcaldia": "002", "nombre_oficial": "Azcapotzalco"},
    {"cve_alcaldia": "003", "nombre_oficial": "Coyoacán"},
    {"cve_alcaldia": "004", "nombre_oficial": "Cuajimalpa de Morelos"},
    {"cve_alcaldia": "005", "nombre_oficial": "Gustavo A. Madero"},
    {"cve_alcaldia": "006", "nombre_oficial": "Iztacalco"},
    {"cve_alcaldia": "007", "nombre_oficial": "Iztapalapa"},
    {"cve_alcaldia": "008", "nombre_oficial": "La Magdalena Contreras"},
    {"cve_alcaldia": "009", "nombre_oficial": "Milpa Alta"},
    {"cve_alcaldia": "010", "nombre_oficial": "Álvaro Obregón"},
    {"cve_alcaldia": "011", "nombre_oficial": "Tláhuac"},
    {"cve_alcaldia": "012", "nombre_oficial": "Tlalpan"},
    {"cve_alcaldia": "013", "nombre_oficial": "Xochimilco"},
    {"cve_alcaldia": "014", "nombre_oficial": "Benito Juárez"},
    {"cve_alcaldia": "015", "nombre_oficial": "Cuauhtémoc"},
    {"cve_alcaldia": "016", "nombre_oficial": "Miguel Hidalgo"},
    {"cve_alcaldia": "017", "nombre_oficial": "Venustiano Carranza"},
])
ALCALDIAS_CDMX["nombre_norm"] = normalizar_texto(ALCALDIAS_CDMX["nombre_oficial"])
print(f"Catálogo canónico: {len(ALCALDIAS_CDMX)} alcaldías")
display(ALCALDIAS_CDMX)

# Diccionario nombre_normalizado -> cve_alcaldia (para mapeos rápidos)
MAP_NOMBRE_A_CVE = dict(zip(ALCALDIAS_CDMX["nombre_norm"], ALCALDIAS_CDMX["cve_alcaldia"]))


def asignar_cve_alcaldia(serie_nombres: pd.Series) -> pd.Series:
    """Mapea nombres de alcaldía (en cualquier capitalización) a su CVE INEGI."""
    return normalizar_texto(serie_nombres).map(MAP_NOMBRE_A_CVE)

Catálogo canónico: 16 alcaldías


,cve_alcaldia,nombre_oficial,nombre_norm
0,002,Azcapotzalco,AZCAPOTZALCO
1,003,Coyoacán,COYOACAN
2,004,Cuajimalpa de Morelos,CUAJIMALPA DE MORELOS
3,005,Gustavo A. Madero,GUSTAVO A. MADERO
4,006,Iztacalco,IZTACALCO
5,007,Iztapalapa,IZTAPALAPA
6,008,La Magdalena Contreras,LA MAGDALENA CONTRERAS
7,009,Milpa Alta,MILPA ALTA
8,010,Álvaro Obregón,ALVARO OBREGON
9,011,Tláhuac,TLAHUAC


## 4. Limpieza por fuente

Cada subsección sigue el mismo patrón: **carga → filtro de scope → tipos → reglas de negocio → selección de columnas → asignación de clave canónica → export**. Al final se imprime el resumen `n_in → n_out` y se registra en `LOG`.

### 4.1 Anuario de Morbilidad SSA 2017

Decisiones del perfilado:
* Encoding `cp1252` confirmado.
* Filtros: `CVE_ESTADO == 9` y diagnóstico ∈ ENFERMEDADES_HIDRICAS → 8 filas.
* Sin nulos, sin duplicados, sin negativos. La regla `ACUMULADO = TOTALM + TOTALF` se cumple al 100 %; **no se requiere ninguna reparación**.
* Acción: seleccionar las 28 columnas relevantes para el proyecto (clave + diagnóstico + grupos de edad ambos sexos + grupos de edad femenino + totales) y normalizar a `snake_case`.

In [9]:
header("4.1 Morbilidad SSA — Limpieza")

df_morbi = pd.read_csv(RUTAS_IN["morbilidad_ssa"], encoding="cp1252", low_memory=False)
n_in = len(df_morbi)

# Filtro CDMX × hídricas
mask = (df_morbi["CVE_ESTADO"] == 9) & (df_morbi["DES_DIAGNO"].isin(ENFERMEDADES_HIDRICAS))
df_morbi = df_morbi.loc[mask].copy()
resumen_paso("Filtrado CDMX × hídricas", n_in, len(df_morbi))

# Selección de columnas relevantes
COLS_BASE       = ["CVE_ESTADO", "DES_ESTADO", "CVE_DIAGNO", "DES_DIAGNO", "ACUMULADO", "PERIODO"]
COLS_EDAD_TOT   = ["MENORES_1", "DE01_A_04", "DE05_A_09", "DE10_A_14", "DE15_A_19",
                   "DE20_A_24", "DE25_A_44", "DE45_A_49", "DE50_A_59", "DE60_A_64",
                   "DE65_Y_MAS", "SE_IGNORAN"]
COLS_EDAD_F     = ["MENORES_1F", "DE01_A_04F", "DE05_A_09F", "DE10_A_14F", "DE15_A_19F",
                   "DE20_A_24F", "DE25_A_44F", "DE45_A_49F", "DE50_A_59F", "DE60_A_64F",
                   "DE65_Y_MAF", "SE_IGNORAF"]
COLS_TOTALES    = ["TOTALM", "TOTALF"]
df_morbi = df_morbi[COLS_BASE + COLS_EDAD_TOT + COLS_EDAD_F + COLS_TOTALES].copy()

# snake_case + normalización de descripciones
df_morbi.columns = [c.lower() for c in df_morbi.columns]
df_morbi["des_estado"] = normalizar_texto(df_morbi["des_estado"])
df_morbi["des_diagno"] = df_morbi["des_diagno"].str.strip()

# Asignación de clave canónica de entidad (no aplica alcaldía: la fuente es estatal)
df_morbi["cve_entidad"] = df_morbi["cve_estado"].astype(str).str.zfill(2)

# Verificación final: la regla ACUMULADO = TOTALM + TOTALF debe seguir cumpliéndose al 100%
ok = (df_morbi["acumulado"] == df_morbi["totalm"] + df_morbi["totalf"]).mean() * 100
assert ok == 100.0, f"Regla de consistencia rota: {ok:.2f}%"
print(f"  ✅ Regla ACUMULADO = TOTALM + TOTALF: 100% de filas")

# Export
df_morbi.to_csv(RUTAS_OUT["morbilidad"], index=False, encoding="utf-8-sig")
LOG["morbilidad_ssa"] = {"n_in": n_in, "n_out": len(df_morbi),
                          "columnas_finales": df_morbi.columns.tolist(),
                          "transformaciones": ["filtro_cdmx", "filtro_hidricas",
                                                "seleccion_columnas", "snake_case"]}
print(f"\n💾 {RUTAS_OUT['morbilidad'].name}: {df_morbi.shape}")
display(df_morbi.head())


 4.1 Morbilidad SSA — Limpieza
  Filtrado CDMX × hídricas                            4,672 →          8  (-4,664 | -99.83%)  
  ✅ Regla ACUMULADO = TOTALM + TOTALF: 100% de filas

💾 morbilidad_cdmx_limpio.csv: (8, 33)


,cve_estado,des_estado,cve_diagno,des_diagno,acumulado,periodo,menores_1,de01_a_04,de05_a_09,de10_a_14,de15_a_19,de20_a_24,de25_a_44,de45_a_49,de50_a_59,de60_a_64,de65_y_mas,se_ignoran,menores_1f,de01_a_04f,de05_a_09f,de10_a_14f,de15_a_19f,de20_a_24f,de25_a_44f,de45_a_49f,de50_a_59f,de60_a_64f,de65_y_maf,se_ignoraf,totalm,totalf,cve_entidad
1168,9,CIUDAD DE MEXICO,2,Amebiasis intestinal,6621,2017,69,347,387,281,173,235,645,174,236,134,232,1,55,359,362,310,219,266,843,300,438,214,341,0,2914,3707,09
1171,9,CIUDAD DE MEXICO,5,Shigelosis,38,2017,0,4,2,2,1,3,2,1,1,1,0,0,0,1,3,2,3,1,5,2,4,0,0,0,17,21,09
1172,9,CIUDAD DE MEXICO,6,Fiebre tifoidea,156,2017,1,0,2,10,8,8,21,14,9,3,2,0,0,0,2,4,15,13,20,9,11,1,3,0,78,78,09
1173,9,CIUDAD DE MEXICO,7,Giardiasis,508,2017,3,51,42,30,11,13,16,13,13,7,16,0,4,53,30,33,12,17,46,14,41,13,30,0,215,293,09
1174,9,CIUDAD DE MEXICO,8,Infecciones intestinales por otros organismos ...,449762,2017,9537,26337,20524,14515,11039,18284,53261,13393,17203,8491,14475,27,9216,24402,19434,13373,11928,19211,60332,18840,28516,13262,24146,16,207086,242676,09


### 4.2 Censo INEGI ITER 2020

Decisiones críticas del perfilado:

1. **Coordenadas en formato DMS** (no decimal). El perfilado reportó 100 % de "nulos" tras la coerción numérica simple — son strings tipo `19°25'42.394"`. Se aplica `parsear_coords_dms`.
2. **Localidades especiales**: 17 filas de totales (`LOC=0`) y 15 confidenciales (`LOC ∈ {9998, 9999}`). Se eliminan: la fila del total ya está implícita en la suma de localidades reales; las confidenciales no aportan información útil al análisis territorial.
3. **Columnas con 22 % de nulos** (POB0_14, VPH_AGUADV, etc.): son localidades pequeñas con confidencialidad estadística (INEGI suprime variables cuando hay ≤ 3 viviendas). Se eliminan esas filas (no se imputa: inventaríamos demografía).
4. Se conservan las 24 columnas relevantes para el proyecto: identificación geográfica, demografía agregada, infraestructura hídrica.

In [10]:
header("4.2 INEGI ITER — Limpieza")

df_iter = pd.read_csv(RUTAS_IN["inegi_iter"], encoding="utf-8-sig", low_memory=False,
                       na_values=MISSING_CODES["INEGI"])
n_in = len(df_iter)

# Filtro CDMX
df_iter["ENTIDAD"] = pd.to_numeric(df_iter["ENTIDAD"], errors="coerce")
df_iter = df_iter[df_iter["ENTIDAD"] == 9].copy()
resumen_paso("Filtro CDMX (ENTIDAD=9)", n_in, len(df_iter))

# Filtro localidades reales
n_pre = len(df_iter)
df_iter = df_iter[(df_iter["LOC"] != 0) & (~df_iter["LOC"].isin([9998, 9999]))].copy()
resumen_paso("Eliminar totales/confidenciales", n_pre, len(df_iter))


 4.2 INEGI ITER — Limpieza
  Filtro CDMX (ENTIDAD=9)                           195,662 →        666  (-194,996 | -99.66%)  
  Eliminar totales/confidenciales                       666 →        634  (-32 | -4.80%)  


In [11]:
# --- Parseo de coordenadas DMS → decimal ---
n_pre_coords = df_iter["LATITUD"].notna().sum()
df_iter["latitud"]  = parsear_coords_dms(df_iter["LATITUD"])
df_iter["longitud"] = parsear_coords_dms(df_iter["LONGITUD"])
n_post_coords = df_iter["latitud"].notna().sum()
print(f"\nParseo de coordenadas DMS:")
print(f"  Filas con coordenada (raw): {n_pre_coords}")
print(f"  Filas con coordenada decimal recuperada: {n_post_coords}")
print(f"  Tasa de recuperación: {100*n_post_coords/max(n_pre_coords,1):.2f}%")

# Validación post-parseo: las coordenadas recuperadas deben caer en CDMX
mask_geo_invalido = (df_iter["latitud"].notna() &
                      ((df_iter["latitud"]  < CDMX_BBOX["lat_min"]) |
                       (df_iter["latitud"]  > CDMX_BBOX["lat_max"]) |
                       (df_iter["longitud"] < CDMX_BBOX["lon_min"]) |
                       (df_iter["longitud"] > CDMX_BBOX["lon_max"])))
print(f"  Coordenadas fuera del bounding box CDMX: {mask_geo_invalido.sum()}")
df_iter.loc[mask_geo_invalido, ["latitud", "longitud"]] = np.nan


Parseo de coordenadas DMS:
  Filas con coordenada (raw): 634
  Filas con coordenada decimal recuperada: 634
  Tasa de recuperación: 100.00%
  Coordenadas fuera del bounding box CDMX: 0


In [12]:
# --- Coerción de variables demográficas y de vivienda ---
COLS_NUM_INEGI = ["POBTOT", "POB0_14", "POB15_64", "POB65_MAS",
                  "VIVTOT", "TVIVPARHAB", "VPH_AGUADV", "VPH_AGUAFV",
                  "VPH_TINACO", "VPH_CISTER", "VPH_DRENAJ", "VPH_NODREN",
                  "VPH_C_SERV", "VPH_EXCSA", "ALTITUD"]
for c in COLS_NUM_INEGI:
    df_iter[c] = pd.to_numeric(df_iter[c], errors="coerce")

# Eliminar filas con confidencialidad estadística (POBTOT presente pero variables agregadas faltantes)
n_pre = len(df_iter)
mask_completo = df_iter[["POBTOT", "POB0_14", "TVIVPARHAB", "VPH_AGUADV"]].notna().all(axis=1)
df_iter = df_iter[mask_completo].copy()
resumen_paso("Eliminar localidades confidenciales", n_pre, len(df_iter),
             "(variables agregadas suprimidas por INEGI)")

# Selección de columnas relevantes y snake_case
COLS_FINALES = ["ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "NOM_LOC",
                "latitud", "longitud", "ALTITUD", "TAMLOC"] + COLS_NUM_INEGI[:-1]
df_iter = df_iter[COLS_FINALES].copy()
df_iter.columns = [c.lower() for c in df_iter.columns]

# Asignación de clave canónica
df_iter["mun"] = df_iter["mun"].astype(int).astype(str).str.zfill(3)
df_iter["cve_alcaldia"] = df_iter["mun"]
df_iter["cve_alcaldia_oficial"] = asignar_cve_alcaldia(df_iter["nom_mun"])
# Verificación: ambas claves deben coincidir (CVE INEGI directa vs nombre)
n_inconsistente = (df_iter["cve_alcaldia"] != df_iter["cve_alcaldia_oficial"]).sum()
print(f"\n  Verificación CVE directa vs nombre: {n_inconsistente} inconsistencias")
df_iter = df_iter.drop(columns=["cve_alcaldia_oficial"])

# Export
df_iter.to_csv(RUTAS_OUT["inegi"], index=False, encoding="utf-8-sig")
LOG["inegi_iter"] = {"n_in": n_in, "n_out": len(df_iter),
                      "columnas_finales": df_iter.columns.tolist(),
                      "transformaciones": ["filtro_cdmx", "filtro_localidades_reales",
                                            "parseo_coords_dms", "coercion_numerica",
                                            "drop_confidenciales", "asignacion_cve_alcaldia"]}
print(f"\n💾 {RUTAS_OUT['inegi'].name}: {df_iter.shape}")
display(df_iter.head(3))

  Eliminar localidades confidenciales                   634 →        494  (-140 | -22.08%)  (variables agregadas suprimidas por INEGI)

  Verificación CVE directa vs nombre: 0 inconsistencias

💾 inegi_cdmx_limpio.csv: (494, 25)


,entidad,nom_ent,mun,nom_mun,loc,nom_loc,latitud,longitud,altitud,tamloc,pobtot,pob0_14,pob15_64,pob65_mas,vivtot,tvivparhab,vph_aguadv,vph_aguafv,vph_tinaco,vph_cister,vph_drenaj,vph_nodren,vph_c_serv,vph_excsa,cve_alcaldia
52276,9,Ciudad de México,002,Azcapotzalco,1,Azcapotzalco,19.484103,-99.184361,2244,12.0,432205,70366.0,306611.0,54863.0,149381,134168.0,133597.0,239.0,107591.0,84062.0,133748.0,87.0,133511.0,133572.0,002
52278,9,Ciudad de México,003,Coyoacán,1,Coyoacán,19.350214,-99.162146,2247,13.0,614447,91327.0,432544.0,90448.0,208024,191517.0,191053.0,133.0,154811.0,114453.0,190986.0,182.0,190814.0,190834.0,003
52280,9,Ciudad de México,004,Cuajimalpa de Morelos,1,Cuajimalpa de Morelos,19.357350,-99.299792,2780,11.0,186693,37821.0,133841.0,14922.0,56986,52524.0,52297.0,124.0,44923.0,21752.0,52288.0,104.0,52163.0,52233.0,004


### 4.3 CONEVAL · Pobreza por grupos de edad

Decisiones del perfilado:
* Sustituir `-999` por `NaN` (código de no calculable).
* `carencia_servicios_basicos_vivienda_porcentaje` tiene 80 % de nulos → **DROP** la columna entera (no es imputable de forma defendible con un 20 % de datos).
* `carencia_calidad_espacios_vivienda_porcentaje` tiene 19 % de nulos → **IMPUTE** con la mediana por (alcaldía, año, grupo de edad).
* Validar rango 0‑100; eliminar las 2 filas con valores fuera del dominio.

In [13]:
header("4.3 CONEVAL — Limpieza")

df_pob = pd.read_csv(RUTAS_IN["coneval_pobreza"], encoding="utf-8")
n_in = len(df_pob)

# Filtro CDMX
df_pob["clave_entidad"] = pd.to_numeric(df_pob["clave_entidad"], errors="coerce")
df_pob = df_pob[df_pob["clave_entidad"] == 9].copy()
resumen_paso("Filtro CDMX (clave_entidad=9)", n_in, len(df_pob))

# Sustituir -999 → NaN
COLS_PCT = [
    "pobreza_porcentaje",
    "carencia_servicios_basicos_vivienda_porcentaje",
    "carencia_calidad_espacios_vivienda_porcentaje",
    "ingreso_inferior_a_lpi_porcentaje",
]
for c in COLS_PCT:
    df_pob[c] = df_pob[c].replace(MISSING_CODES["CONEVAL"], np.nan)

# Validar rango 0-100
n_pre = len(df_pob)
mask_invalido = ((df_pob[COLS_PCT] < 0) | (df_pob[COLS_PCT] > 100)).any(axis=1)
df_pob = df_pob[~mask_invalido].copy()
resumen_paso("Eliminar % fuera del rango 0-100", n_pre, len(df_pob))


 4.3 CONEVAL — Limpieza
  Filtro CDMX (clave_entidad=9)                      29,528 →        192  (-29,336 | -99.35%)  
  Eliminar % fuera del rango 0-100                      192 →        190  (-2 | -1.04%)  


In [14]:
# DROP columna con 80% de nulos
df_pob = df_pob.drop(columns=["carencia_servicios_basicos_vivienda_porcentaje"])
print("  ✂️  DROP carencia_servicios_basicos_vivienda_porcentaje (80% nulos)")

# IMPUTE carencia_calidad_espacios con mediana por (municipio, año, grupo)
df_pob["anio"] = pd.to_datetime(df_pob["periodo"], errors="coerce").dt.year
n_pre_imp = df_pob["carencia_calidad_espacios_vivienda_porcentaje"].isna().sum()
df_pob["carencia_calidad_espacios_vivienda_porcentaje"] = (
    df_pob.groupby(["municipio", "anio", "grupo"])["carencia_calidad_espacios_vivienda_porcentaje"]
          .transform(lambda s: s.fillna(s.median()))
)
# Fallback: si quedan NaN tras el groupby, usar mediana global
df_pob["carencia_calidad_espacios_vivienda_porcentaje"] = (
    df_pob["carencia_calidad_espacios_vivienda_porcentaje"]
       .fillna(df_pob["carencia_calidad_espacios_vivienda_porcentaje"].median())
)
n_post_imp = df_pob["carencia_calidad_espacios_vivienda_porcentaje"].isna().sum()
print(f"  🔧 IMPUTE carencia_calidad_espacios: {n_pre_imp} NaN → {n_post_imp} NaN")

# Asignación de clave canónica
df_pob["cve_alcaldia"] = asignar_cve_alcaldia(df_pob["municipio"])
n_sin_cve = df_pob["cve_alcaldia"].isna().sum()
print(f"  Filas sin CVE alcaldía asignada: {n_sin_cve}")

# Selección final de columnas
COLS_FINAL_CONEVAL = ["clave_entidad", "entidad", "clave_municipio", "municipio",
                      "cve_alcaldia", "anio", "periodo", "grupo", "poblacion",
                      "pobreza_porcentaje",
                      "carencia_calidad_espacios_vivienda_porcentaje",
                      "ingreso_inferior_a_lpi_porcentaje"]
COLS_FINAL_CONEVAL = [c for c in COLS_FINAL_CONEVAL if c in df_pob.columns]
df_pob = df_pob[COLS_FINAL_CONEVAL].copy()

# Export
df_pob.to_csv(RUTAS_OUT["coneval"], index=False, encoding="utf-8-sig")
LOG["coneval_pobreza"] = {"n_in": n_in, "n_out": len(df_pob),
                           "columnas_finales": df_pob.columns.tolist(),
                           "transformaciones": ["filtro_cdmx", "reemplazo_-999",
                                                 "filtro_rango_0_100", "drop_carencia_servicios",
                                                 "impute_calidad_espacios", "cve_alcaldia"]}
print(f"\n💾 {RUTAS_OUT['coneval'].name}: {df_pob.shape}")
display(df_pob.head(3))

  ✂️  DROP carencia_servicios_basicos_vivienda_porcentaje (80% nulos)
  🔧 IMPUTE carencia_calidad_espacios: 35 NaN → 0 NaN
  Filas sin CVE alcaldía asignada: 0

💾 coneval_cdmx_limpio.csv: (190, 12)


,clave_entidad,entidad,clave_municipio,municipio,cve_alcaldia,anio,periodo,grupo,poblacion,pobreza_porcentaje,carencia_calidad_espacios_vivienda_porcentaje,ingreso_inferior_a_lpi_porcentaje
273,9,Ciudad de México,9002,Azcapotzalco,002,2020,2020-01-01,"Niñas, niños y adolescentes (0 a 17 años)",86146,33.560905,5.716876,46.758250
274,9,Ciudad de México,9003,Coyoacán,003,2020,2020-01-01,"Niñas, niños y adolescentes (0 a 17 años)",110711,37.442645,5.571868,50.530831
275,9,Ciudad de México,9004,Cuajimalpa de Morelos,004,2020,2020-01-01,"Niñas, niños y adolescentes (0 a 17 años)",49710,43.110548,9.453873,54.066137


### 4.4 Catálogo SEPOMEX

Decisiones del perfilado:
* Score Validez = 37.4 % por **956 CPs con formato inválido**. La causa probable: al cargar como string algunos CPs perdieron el `0` inicial (CDMX usa rango `01000`‑`16999`). Solución: extraer dígitos, hacer `zfill(5)`, validar rango.
* Sin duplicados; sin CPs multi‑estado.
* Asignación de `cve_alcaldia` canónica para joins.

In [15]:
header("4.4 SEPOMEX — Limpieza")

df_sep = pd.read_csv(RUTAS_IN["sepomex"], encoding="utf-8-sig",
                      dtype={"codigo_postal": str, "cve_estado": str, "cve_alcaldia": str})
n_in = len(df_sep)

# Reparar CP: extraer dígitos, zfill, validar rango CDMX
def normalizar_cp(s: pd.Series) -> pd.Series:
    cp = s.astype(str).str.extract(r"(\d+)", expand=False)
    return cp.str.zfill(5)

df_sep["codigo_postal"] = normalizar_cp(df_sep["codigo_postal"])

# Validación: CP de CDMX debe estar en 01000-16999
cp_num = pd.to_numeric(df_sep["codigo_postal"], errors="coerce")
mask_cp_valido = cp_num.between(1000, 16999)
n_invalidos = (~mask_cp_valido).sum()
print(f"  CPs con formato/rango inválido tras reparación: {n_invalidos}")

# Decisión: filtramos los inválidos (no podemos arreglar lo que no es CP)
n_pre = len(df_sep)
df_sep = df_sep[mask_cp_valido].copy()
resumen_paso("Eliminar CPs fuera de rango CDMX", n_pre, len(df_sep))

# Padding de claves
df_sep["cve_estado"]   = df_sep["cve_estado"].astype(str).str.zfill(2)
df_sep["cve_alcaldia"] = df_sep["cve_alcaldia"].astype(str).str.zfill(3)

# Verificación: cve_alcaldia debe coincidir con el catálogo canónico
df_sep["cve_alcaldia_oficial"] = asignar_cve_alcaldia(df_sep["nom_alcaldia"])
n_inconsistente = (df_sep["cve_alcaldia"] != df_sep["cve_alcaldia_oficial"]).sum()
print(f"  Inconsistencias CVE directa vs nombre: {n_inconsistente}")
df_sep = df_sep.drop(columns=["cve_alcaldia_oficial"])

# Normalización de nombres (preservamos el nombre original; agregamos uno normalizado)
df_sep["nom_alcaldia_norm"] = normalizar_texto(df_sep["nom_alcaldia"])
df_sep["nom_colonia_norm"]  = normalizar_texto(df_sep["nom_colonia"])

# Export
df_sep.to_csv(RUTAS_OUT["sepomex"], index=False, encoding="utf-8-sig")
LOG["sepomex"] = {"n_in": n_in, "n_out": len(df_sep),
                   "columnas_finales": df_sep.columns.tolist(),
                   "transformaciones": ["normalizar_cp", "filtro_cp_cdmx",
                                         "padding_claves", "norm_alcaldia_colonia"]}
print(f"\n💾 {RUTAS_OUT['sepomex'].name}: {df_sep.shape}")
display(df_sep.head(3))


 4.4 SEPOMEX — Limpieza
  CPs con formato/rango inválido tras reparación: 0
  Eliminar CPs fuera de rango CDMX                    1,526 →      1,526  (-0 | -0.00%)  
  Inconsistencias CVE directa vs nombre: 0

💾 sepomex_cdmx_limpio.csv: (1526, 8)


,nom_estado,cve_estado,nom_alcaldia,cve_alcaldia,nom_colonia,codigo_postal,nom_alcaldia_norm,nom_colonia_norm
0,Ciudad de México,09,Álvaro Obregón,010,San Ángel,01000,ALVARO OBREGON,SAN ANGEL
1,Ciudad de México,09,Álvaro Obregón,010,Los Alpes,01010,ALVARO OBREGON,LOS ALPES
2,Ciudad de México,09,Álvaro Obregón,010,Guadalupe Inn,01020,ALVARO OBREGON,GUADALUPE INN


### 4.5 SACMEX · Reportes de fugas

La sección más voluminosa porque concentra los hallazgos críticos:

| Hallazgo del perfilado | Decisión |
|---|---|
| 189 123 fechas no parseables con `dayfirst=True` | Cascada `parsear_fecha_robusta` (7 formatos) |
| 14 654 lag negativo (reporte antes que incidente) | DROP — imposible físicamente |
| 4 946 lag > 30 días | KEEP con flag `lag_largo` (pueden ser legítimos) |
| 12 729 duplicados de fila completa | DROP |
| 58 610 duplicados de `folio_incidente` | KEEP — son updates legítimos del mismo incidente |
| 12 729 duplicados de `id_reporte` | Coinciden con duplicados completos → ya removidos |
| 166 coordenadas fuera del bbox CDMX | KEEP con flag `geo_fuera_cdmx` |
| 60 843 colonias sin match en SEPOMEX (19.4 %) | KEEP — record‑linkage avanzado se hace en `fusion.ipynb` |

> **Alerta de scope (del perfilado):** sólo cubre ene‑2024. El histórico 2018‑2022 debe agregarse antes de `fusion.ipynb`. Esta limpieza está diseñada para encadenar varios CSVs SACMEX sin tocar.

In [16]:
header("4.5 SACMEX — Limpieza (concatenación hist + actual)")

# --- Carga del SACMEX actual 2022-2024 ---
df_sac_actual = pd.read_csv(RUTAS_IN["sacmex_reportes"], encoding="utf-8", low_memory=False,
                             dtype={"folio_incidente": str, "id_reporte": str})
print(f"  SACMEX actual (2022-2024): {len(df_sac_actual):,} filas")

# --- Carga del SACMEX histórico 2018-2021 (NUEVO v3) ---
if RUTAS_IN["sacmex_hist"].exists():
    df_sac_hist = pd.read_csv(RUTAS_IN["sacmex_hist"], encoding="utf-8", low_memory=False,
                                dtype={"folio": str, "codigo_postal": str})
    print(f"  SACMEX histórico (2018-2021): {len(df_sac_hist):,} filas")

    # --- Mapeo de schema: hist → actual ---
    # El histórico usa nombres distintos; armonizamos para concatenar
    df_sac_hist_renombrado = df_sac_hist.rename(columns={
        "folio":                   "folio_incidente",
        "tipo_de_falla":           "clasificacion",
        "fecha":                   "fecha_reporte",        # única; lag será NaN
        "colonia_registro_sacmex": "colonia_catalogo",
        "alcaldia":                "alcaldia_catalogo",
    }).copy()

    # Columnas que existen sólo en el histórico
    if "codigo_postal" not in df_sac_hist_renombrado.columns:
        df_sac_hist_renombrado["codigo_postal"] = pd.NA

    # Columnas que existen sólo en el actual (rellenamos con NaN)
    cols_actual_only = ["id_reporte", "fecha_registro_incidente", "reporte",
                        "medio_recepcion", "hora_reporte"]
    for c in cols_actual_only:
        if c in df_sac_actual.columns and c not in df_sac_hist_renombrado.columns:
            df_sac_hist_renombrado[c] = pd.NA

    # Descartar columnas históricas que no aportan al análisis
    cols_descartar = ["quien_atiende", "colonia_datos_abiertos"]
    df_sac_hist_renombrado = df_sac_hist_renombrado.drop(
        columns=[c for c in cols_descartar if c in df_sac_hist_renombrado.columns]
    )

    # Marcar origen de cada registro (auditoría)
    df_sac_hist_renombrado["origen_dataset"] = "hist_2018_2021"
    df_sac_actual["origen_dataset"] = "actual_2022_2024"

    # --- Concatenar (las columnas que falten en uno se vuelven NaN) ---
    df_sac = pd.concat([df_sac_hist_renombrado, df_sac_actual], ignore_index=True, sort=False)

    print(f"\n  Total tras concatenación: {len(df_sac):,} filas")
    print(f"  Distribución por origen:")
    print(df_sac["origen_dataset"].value_counts())
else:
    print("  ⚠️ reportes_agua_hist.csv NO encontrado; sólo se procesará el actual")
    df_sac_actual["origen_dataset"] = "actual_2022_2024"
    df_sac = df_sac_actual

n_in = len(df_sac)
print(f"\n  Carga inicial total: {n_in:,} filas")

# 1) Eliminar duplicados de fila completa
n_pre = len(df_sac)
df_sac = df_sac.drop_duplicates().copy()
resumen_paso("Drop duplicados de fila completa", n_pre, len(df_sac))


 4.5 SACMEX — Limpieza (concatenación hist + actual)
  SACMEX actual (2022-2024): 313,756 filas
  SACMEX histórico (2018-2021): 254,730 filas

  Total tras concatenación: 568,486 filas
  Distribución por origen:
origen_dataset
actual_2022_2024    313756
hist_2018_2021      254730
Name: count, dtype: int64

  Carga inicial total: 568,486 filas
  Drop duplicados de fila completa                  568,486 →    555,757  (-12,729 | -2.24%)  


In [17]:
# 2) Parseo robusto de fechas (cascada de formatos)
print("\n--- Parseo de fechas ---")
df_sac["fecha_registro_dt"] = parsear_fecha_robusta(df_sac["fecha_registro_incidente"])
df_sac["fecha_reporte_dt"]  = parsear_fecha_robusta(df_sac["fecha_reporte"])

n_no_parseado_reg = df_sac["fecha_registro_dt"].isna().sum()
n_no_parseado_rep = df_sac["fecha_reporte_dt"].isna().sum()
print(f"  Fechas no parseables (fecha_registro_incidente): {n_no_parseado_reg:,}")
print(f"  Fechas no parseables (fecha_reporte):           {n_no_parseado_rep:,}")
print(f"  Rango fechas válidas: {df_sac['fecha_reporte_dt'].min()}  →  {df_sac['fecha_reporte_dt'].max()}")

# 3) Eliminar filas sin fecha de reporte (sin la métrica clave del proyecto, no aportan)
n_pre = len(df_sac)
df_sac = df_sac[df_sac["fecha_reporte_dt"].notna()].copy()
resumen_paso("Drop filas sin fecha_reporte parseable", n_pre, len(df_sac))


--- Parseo de fechas ---
  Fechas no parseables (fecha_registro_incidente): 254,730
  Fechas no parseables (fecha_reporte):           0
  Rango fechas válidas: 2018-01-01 00:00:00  →  2024-02-01 00:00:00
  Drop filas sin fecha_reporte parseable            555,757 →    555,757  (-0 | -0.00%)  


In [18]:
# 4) Calcular lag y filtrar lag negativo
df_sac["lag_dias"] = (df_sac["fecha_reporte_dt"] - df_sac["fecha_registro_dt"]).dt.total_seconds() / 86400

n_pre = len(df_sac)
df_sac = df_sac[(df_sac["lag_dias"].isna()) | (df_sac["lag_dias"] >= 0)].copy()
resumen_paso("Drop lag negativo (imposible)", n_pre, len(df_sac))

# 5) Marcar lag largo (>30 días) como flag de calidad, no eliminar
df_sac["flag_lag_largo"] = (df_sac["lag_dias"] > 30).astype(int)
print(f"  🚩 flag_lag_largo = 1: {df_sac['flag_lag_largo'].sum():,} filas")

# 6) Filtrar al rango temporal del proyecto (2018-2024 según README)
n_pre = len(df_sac)
mask_rango = df_sac["fecha_reporte_dt"].between(pd.Timestamp("2018-01-01"),
                                                  pd.Timestamp("2024-12-31"))
df_sac = df_sac[mask_rango].copy()
resumen_paso("Filtrar fechas 2018-2024", n_pre, len(df_sac))

  Drop lag negativo (imposible)                     555,757 →    519,033  (-36,724 | -6.61%)  
  🚩 flag_lag_largo = 1: 4,182 filas
  Filtrar fechas 2018-2024                          519,033 →    519,033  (-0 | -0.00%)  


In [19]:
# 7) Validez geográfica: marcar fuera de bbox sin eliminar
df_sac["latitud"]  = pd.to_numeric(df_sac["latitud"],  errors="coerce")
df_sac["longitud"] = pd.to_numeric(df_sac["longitud"], errors="coerce")

mask_geo_invalido = (df_sac["latitud"].notna() & df_sac["longitud"].notna() &
                      ((df_sac["latitud"]  < CDMX_BBOX["lat_min"]) |
                       (df_sac["latitud"]  > CDMX_BBOX["lat_max"]) |
                       (df_sac["longitud"] < CDMX_BBOX["lon_min"]) |
                       (df_sac["longitud"] > CDMX_BBOX["lon_max"])))
df_sac["flag_geo_fuera_cdmx"] = mask_geo_invalido.astype(int)
print(f"  🚩 flag_geo_fuera_cdmx = 1: {df_sac['flag_geo_fuera_cdmx'].sum():,} filas")

# 8) Normalizar texto de alcaldía y colonia + asignar clave canónica
df_sac["alcaldia_norm"] = normalizar_texto(df_sac["alcaldia_catalogo"])
df_sac["colonia_norm"]  = normalizar_texto(df_sac["colonia_catalogo"])
df_sac["cve_alcaldia"]  = df_sac["alcaldia_norm"].map(MAP_NOMBRE_A_CVE)
n_sin_cve = df_sac["cve_alcaldia"].isna().sum()
print(f"  Filas sin cve_alcaldia (alcaldía no canónica): {n_sin_cve:,}")

# 9) Hora del reporte parseada
df_sac["hora_reporte_dt"] = pd.to_datetime(df_sac["hora_reporte"], errors="coerce", format="%H:%M:%S").dt.time

# 10) Selección y export
COLS_FINAL_SAC = ["folio_incidente", "id_reporte",
                  "fecha_registro_dt", "fecha_reporte_dt", "hora_reporte_dt",
                  "lag_dias", "flag_lag_largo",
                  "clasificacion", "reporte", "medio_recepcion",
                  "alcaldia_catalogo", "alcaldia_norm", "cve_alcaldia",
                  "colonia_catalogo", "colonia_norm",
                  "latitud", "longitud", "flag_geo_fuera_cdmx"]
df_sac = df_sac[COLS_FINAL_SAC].rename(columns={
    "fecha_registro_dt": "fecha_registro_incidente",
    "fecha_reporte_dt": "fecha_reporte",
    "hora_reporte_dt": "hora_reporte",
})

df_sac.to_csv(RUTAS_OUT["sacmex"], index=False, encoding="utf-8-sig")
LOG["sacmex_reportes"] = {"n_in": n_in, "n_out": len(df_sac),
                           "columnas_finales": df_sac.columns.tolist(),
                           "transformaciones": ["drop_duplicados_completos",
                                                 "parseo_fecha_robusto",
                                                 "drop_sin_fecha_reporte",
                                                 "drop_lag_negativo",
                                                 "flag_lag_largo",
                                                 "filtro_2018_2024",
                                                 "flag_geo_fuera_cdmx",
                                                 "norm_texto_y_cve_alcaldia"]}
print(f"\n💾 {RUTAS_OUT['sacmex'].name}: {df_sac.shape}")
display(df_sac.head(3))

  🚩 flag_geo_fuera_cdmx = 1: 2,344 filas
  Filas sin cve_alcaldia (alcaldía no canónica): 4,873

💾 sacmex_cdmx_limpio.csv: (519033, 18)


,folio_incidente,id_reporte,fecha_registro_incidente,fecha_reporte,hora_reporte,lag_dias,flag_lag_largo,clasificacion,reporte,medio_recepcion,alcaldia_catalogo,alcaldia_norm,cve_alcaldia,colonia_catalogo,colonia_norm,latitud,longitud,flag_geo_fuera_cdmx
0,20210405-0030,<NA>,NaT,2021-04-05,NaT,NaN,0,Falta de agua,<NA>,<NA>,ALVARO OBREGON,ALVARO OBREGON,010,Alfonso XIII,ALFONSO XIII,19.378935,-99.191926,0
1,20210428-0107,<NA>,NaT,2021-04-28,NaT,NaN,0,Falta de agua,<NA>,<NA>,ALVARO OBREGON,ALVARO OBREGON,010,Olivar de los Padres,OLIVAR DE LOS PADRES,19.342617,-99.218876,0
2,20210428-0184,<NA>,NaT,2021-04-28,NaT,NaN,0,Falta de agua,<NA>,<NA>,ALVARO OBREGON,ALVARO OBREGON,010,Lomas de los Ángeles Tetelpan,LOMAS DE LOS ANGELES TETELPAN,19.342893,-99.218000,0


### 4.6 CONAGUA Sitios

Decisiones del perfilado:
* Catálogo nacional limpio (sin duplicados de `CLAVE SITIO`).
* 15 sitios fuera del bbox de México → DROP (errores de captura).
* Las columnas `CLAVE ACUÍFERO`, `ACUÍFERO`, `CUENCA` con 35‑63 % de nulos: son **nulos por diseño** (un sitio superficial no tiene acuífero asociado). Se conservan.
* **Conservamos el catálogo nacional completo**, no filtramos a CDMX, porque los sitios que abastecen a CDMX (Cutzamala, Lerma) están en EdoMex / Michoacán. La fusión territorial se hace en `fusion.ipynb` con join a Resultados+CDMX.

In [20]:
header("4.6 CONAGUA Sitios — Limpieza")

df_sit = pd.read_csv(RUTAS_IN["conagua_sitios"], encoding="utf-8-sig", low_memory=False)
n_in = len(df_sit)

# Coerción de coordenadas
df_sit["LATITUD"]  = pd.to_numeric(df_sit["LATITUD"],  errors="coerce")
df_sit["LONGITUD"] = pd.to_numeric(df_sit["LONGITUD"], errors="coerce")

# Filtrar a territorio mexicano
n_pre = len(df_sit)
mask_mx = df_sit["LATITUD"].between(MX_BBOX["lat_min"], MX_BBOX["lat_max"]) &           df_sit["LONGITUD"].between(MX_BBOX["lon_min"], MX_BBOX["lon_max"])
df_sit = df_sit[mask_mx].copy()
resumen_paso("Filtrar a territorio mexicano", n_pre, len(df_sit))

# Marcar si el sitio está en CDMX (para joins)
df_sit["estado_norm"] = normalizar_texto(df_sit["ESTADO"])
df_sit["es_cdmx"] = df_sit["estado_norm"].isin(["CIUDAD DE MEXICO", "DISTRITO FEDERAL"]).astype(int)
print(f"  Sitios en CDMX: {df_sit['es_cdmx'].sum()}")

# snake_case
df_sit.columns = [c.lower().replace(" ", "_").replace("á","a").replace("í","i").replace("ó","o")
                   for c in df_sit.columns]

df_sit.to_csv(RUTAS_OUT["conagua_sitios"], index=False, encoding="utf-8-sig")
LOG["conagua_sitios"] = {"n_in": n_in, "n_out": len(df_sit),
                          "columnas_finales": df_sit.columns.tolist(),
                          "transformaciones": ["coercion_coords", "filtro_mexico",
                                                "flag_es_cdmx", "snake_case"]}
print(f"\n💾 {RUTAS_OUT['conagua_sitios'].name}: {df_sit.shape}")
display(df_sit.head(3))


 4.6 CONAGUA Sitios — Limpieza
  Filtrar a territorio mexicano                       7,771 →      7,756  (-15 | -0.19%)  
  Sitios en CDMX: 20

💾 conagua_sitios_limpio.csv: (7756, 16)


,clave_sitio,nombre_del_sitio,cuenca,clave_acuifero,acuifero,organismo_cuenca,direccion_local,estado,municipio,cuerpo_de_agua,tipo_de_cuerpo_de_agua,subtipo_cuerpo_agua,latitud,longitud,estado_norm,es_cdmx
0,OCPYU4929,COSGAYA,NaN,3105.0,PENÍNSULA DE YUCATÁN,PENÍNSULA DE YUCATÁN,NaN,YUCATÁN,MÉRIDA,ACUÍFERO PENÍNSULA DE YUCATÁN,SUBTERRÁNEO (ESTUDIO ESPECIAL),POZO,21.09699,-89.70460,YUCATAN,0
1,OCPYU4946,POZO 1 DEL SISTEMA DE AGUA POTABLE DE SOTUTA,NaN,3105.0,PENÍNSULA DE YUCATÁN,PENÍNSULA DE YUCATÁN,NaN,YUCATÁN,SOTUTA,ACUÍFERO PENÍNSULA DE YUCATÁN,SUBTERRÁNEO (ESTUDIO ESPECIAL),POZO,20.59704,-89.00757,YUCATAN,0
2,OCPYU4953W1,ANILLO DE CENOTES DE YUCATAN 5,NaN,3105.0,PENÍNSULA DE YUCATÁN,PENÍNSULA DE YUCATÁN,NaN,YUCATÁN,KOPOMÁ,ACUÍFERO PENÍNSULA DE YUCATÁN,SUBTERRÁNEO (ESTUDIO ESPECIAL),CENOTE,20.68943,-89.87605,YUCATAN,0


### 4.7 CONAGUA Resultados (la cirugía mayor)

> **Cambio de scope respecto a v1:** la versión anterior filtraba sólo por *frontera política* (`ESTADO == 'CIUDAD DE MÉXICO'`), lo que recuperaba 22 sitios locales (Xochimilco, Río Magdalena, Cerro de la Estrella, etc.) pero **dejaba fuera las fuentes que abastecen agua potable a la ciudad**, ubicadas físicamente en otros estados. La CDMX recibe ~26 % de su agua del Sistema Cutzamala (EdoMex y Michoacán), una porción adicional de la Presa Madín (EdoMex) y comparte el acuífero del Valle de México con el Sistema Lerma (EdoMex). Filtrar por estado los excluía.

**Nuevo criterio de scope: pertenencia al sistema hidráulico de CDMX**, no al polígono político. Un sitio entra al dataset limpio si pertenece a cualquiera de estas categorías:

| Subsistema | Justificación | Filtro |
|---|---|---|
| `cdmx_local` | Cuerpos de agua dentro de la frontera política | `ESTADO == 'CIUDAD DE MÉXICO'` |
| `cutzamala_valle_bravo` | Vaso principal Cutzamala (~50 % del aporte del sistema) | Nombre contiene 'VALLE DE BRAVO' |
| `cutzamala_villa_victoria` | Vaso secundario Cutzamala | Nombre contiene 'VILLA VICTORIA' |
| `cutzamala_colorines` | Vaso de bombeo Cutzamala | Nombre contiene 'COLORINES' |
| `cutzamala_el_bosque` | Vaso Cutzamala en Michoacán | Nombre contiene 'PRESA EL BOSQUE' |
| `cutzamala_tuxpan` | Vaso Cutzamala en Michoacán (Tuxpan, no Tuxpango) | Nombre contiene 'PRESA TUXPAN' / 'RIO TUXPAN' / 'CANAL TUXPAN' |
| `cutzamala_rio` | Río Cutzamala | Nombre contiene 'CUTZAMALA' |
| `madin` | Presa Madín — abastece zona norte CDMX vía Naucalpan | Nombre contiene 'MADÍN' / 'MADIN' |
| `lerma` | Acuífero compartido ZMVM — Sistema Lerma | Nombre contiene 'RÍO LERMA' / 'CIÉNEGAS DE LERMA' |

Cada sitio recibe una etiqueta `clasificacion_cdmx` con el subsistema al que pertenece, lo que permite a `fusion.ipynb` agregar/separar análisis por origen del agua (cuerpo local vs. abastecedor importado).

Decisiones del perfilado que se conservan:
* Cruce con catálogo de Sitios (100 % match: 0 huérfanos en el perfilado).
* Tabla de decisión por columna: DROP > 95 % nulos, FLAG 30‑95 %, IMPUTE 5‑30 % vía KNN, KEEP < 5 %.
* Parsear columnas químicas con `<`, `>`, `ND` (convención EPA: L/2 para below_LOD).
* Filtrar filas con `DBO > DQO` (imposible físicamente).

In [21]:
header("4.7 CONAGUA Resultados — Limpieza")

df_res = pd.read_csv(RUTAS_IN["conagua_result"], encoding="utf-8-sig", low_memory=False)
df_sitios_raw = pd.read_csv(RUTAS_IN["conagua_sitios"], encoding="utf-8-sig", low_memory=False)
n_in = len(df_res)

# Cruce con sitios para obtener ESTADO y nombre del sitio
df_res = df_res.merge(
    df_sitios_raw[["CLAVE SITIO", "NOMBRE DEL SITIO", "ESTADO", "LATITUD", "LONGITUD"]],
    on="CLAVE SITIO", how="left", suffixes=("", "_sitio")
)

# ---------------------------------------------------------------------
# Definición de scope: sistema hidráulico de CDMX (no frontera política)
# Cada subsistema = (regex_nombre, lista_estados_validos | None).
# Si estados_validos es None, no se filtra por estado (el nombre es suficiente).
# ---------------------------------------------------------------------
SISTEMAS_ABASTECEDORES = {
    "cdmx_local": {
        "patron":   None,                                       # se aplica por estado, no por nombre
        "estados":  ["CIUDAD DE MEXICO", "DISTRITO FEDERAL"],
    },
    "cutzamala_valle_bravo": {
        "patron":   r"VALLE DE BRAVO",
        "estados":  ["MEXICO"],
    },
    "cutzamala_villa_victoria": {
        "patron":   r"VILLA VICTORIA",
        "estados":  ["MEXICO"],
    },
    "cutzamala_colorines": {
        "patron":   r"COLORINES",
        "estados":  ["MEXICO"],
    },
    "cutzamala_el_bosque": {
        "patron":   r"PRESA EL BOSQUE|INFLUENTE PRESA EL BOSQUE",
        "estados":  ["MICHOACAN"],
    },
    "cutzamala_tuxpan": {
        # excluimos 'TUXPANGO' (Veracruz) con un negative lookahead
        "patron":   r"(?:PRESA TUXPAN(?!GO)|R[ÍI]O TUXPAN|CANAL TUXPAN|LAGO DE TUXPAN)",
        "estados":  ["MICHOACAN", "MEXICO", "GUERRERO"],
    },
    "cutzamala_rio": {
        "patron":   r"CUTZAMALA",
        "estados":  None,                                        # cualquier estado
    },
    "madin": {
        "patron":   r"MAD[ÍI]N",
        "estados":  ["MEXICO"],
    },
    "lerma": {
        "patron":   r"R[ÍI]O LERMA|CI[ÉE]NEGAS DE LERMA",
        "estados":  ["MEXICO"],
    },
}

# Construir la columna de clasificación
estado_norm = normalizar_texto(df_res["ESTADO"])
nombre_norm = normalizar_texto(df_res["NOMBRE DEL SITIO"])
df_res["clasificacion_cdmx"] = pd.NA  # default = no relevante

# Aplicar reglas en orden (la primera que coincide gana)
for subsistema, regla in SISTEMAS_ABASTECEDORES.items():
    sin_clasificar = df_res["clasificacion_cdmx"].isna()
    if regla["patron"] is None:
        # Sólo filtro por estado (caso CDMX local)
        match = sin_clasificar & estado_norm.isin(regla["estados"])
    else:
        match_nombre = nombre_norm.str.contains(regla["patron"], regex=True, na=False)
        if regla["estados"] is None:
            match = sin_clasificar & match_nombre
        else:
            match = sin_clasificar & match_nombre & estado_norm.isin(regla["estados"])
    df_res.loc[match, "clasificacion_cdmx"] = subsistema

# Filtrar al scope (descartar lo que no está clasificado)
df_res = df_res[df_res["clasificacion_cdmx"].notna()].copy()
resumen_paso("Filtrar al sistema hidráulico CDMX", n_in, len(df_res))

# Resumen por subsistema (mediciones + sitios únicos)
resumen_scope = (df_res.groupby("clasificacion_cdmx")
                 .agg(n_mediciones=("CLAVE SITIO", "size"),
                      n_sitios=("CLAVE SITIO", "nunique"),
                      anios=("Año", lambda s: f"{int(s.min())}-{int(s.max())}"))
                 .sort_values("n_mediciones", ascending=False))
print()
display(resumen_scope)
print(f"\n  Total sitios únicos en scope: {df_res['CLAVE SITIO'].nunique()}")
print(f"  Total mediciones en scope:    {len(df_res):,}")



 4.7 CONAGUA Resultados — Limpieza
  Filtrar al sistema hidráulico CDMX                129,235 →        376  (-128,859 | -99.71%)  



,n_mediciones,n_sitios,anios
clasificacion_cdmx,,,
cdmx_local,279,20,2012-2024
cutzamala_rio,69,3,2012-2020
cutzamala_tuxpan,28,2,2012-2022



  Total sitios únicos en scope: 25
  Total mediciones en scope:    376


In [22]:
# Construir tabla de decisión por columna
nulos_pct = df_res.isnull().mean() * 100

def asignar_accion(p):
    if p > DECISION_THRESHOLDS["DROP"]:   return "DROP"
    if p > DECISION_THRESHOLDS["FLAG"]:   return "FLAG"
    if p > DECISION_THRESHOLDS["IMPUTE"]: return "IMPUTE"
    return "KEEP"

catalogo_cols = pd.DataFrame({
    "columna": nulos_pct.index,
    "pct_nulos": nulos_pct.values.round(2),
})
catalogo_cols["accion"] = catalogo_cols["pct_nulos"].apply(asignar_accion)

cols_drop   = catalogo_cols[catalogo_cols["accion"] == "DROP"]["columna"].tolist()
cols_flag   = catalogo_cols[catalogo_cols["accion"] == "FLAG"]["columna"].tolist()
cols_impute = catalogo_cols[catalogo_cols["accion"] == "IMPUTE"]["columna"].tolist()
cols_keep   = catalogo_cols[catalogo_cols["accion"] == "KEEP"]["columna"].tolist()

print(f"  KEEP   : {len(cols_keep)}")
print(f"  IMPUTE : {len(cols_impute)}")
print(f"  FLAG   : {len(cols_flag)}")
print(f"  DROP   : {len(cols_drop)}")

# Aplicar DROP
n_cols_pre = df_res.shape[1]
df_res = df_res.drop(columns=cols_drop)
print(f"  Columnas: {n_cols_pre} → {df_res.shape[1]}  (-{len(cols_drop)})")

# Guardamos la tabla de decisión como artefacto
catalogo_cols.to_csv(DATOS_LIMPIOS / "_catalogo_columnas_conagua.csv",
                     index=False, encoding="utf-8-sig")

  KEEP   : 13
  IMPUTE : 27
  FLAG   : 54
  DROP   : 380
  Columnas: 474 → 94  (-380)


In [23]:
# Parsear columnas químicas con anotaciones (<, >, ND)
COLS_METADATA = ["CLAVE SITIO", "CLAVE DE MONITOREO", "NOMBRE DEL SITIO",
                 "TIPO CUERPO DE AGUA", "FECHA REALIZACIÓN", "Año",
                 "ESTADO", "LATITUD", "LONGITUD"]
cols_quimicas = [c for c in df_res.columns if c not in COLS_METADATA]

print(f"\n--- Parseo químico (<, >, ND) en {len(cols_quimicas)} columnas ---")
n_anotaciones_total = 0
for col in cols_quimicas:
    if df_res[col].dtype == object:
        valor, flag = parsear_medicion_quimica(df_res[col])
        n_anotadas = (flag.isin(["below_LOD", "above_LOQ", "ND"])).sum()
        if n_anotadas > 0:
            df_res[col] = valor
            df_res[f"{col}__flag"] = flag
            n_anotaciones_total += n_anotadas
print(f"  Mediciones con anotación parseadas: {n_anotaciones_total:,}")


--- Parseo químico (<, >, ND) en 85 columnas ---
  Mediciones con anotación parseadas: 0


In [24]:
# --- Aplicar regla DBO ≤ DQO (eliminar filas violadoras) ---
if {"DBO_TOT", "DQO_TOT"}.issubset(df_res.columns):
    n_pre = len(df_res)
    mask_eval = df_res["DBO_TOT"].notna() & df_res["DQO_TOT"].notna()
    mask_violacion = mask_eval & (df_res["DBO_TOT"] > df_res["DQO_TOT"])
    df_res = df_res[~mask_violacion].copy()
    resumen_paso("Drop filas con DBO > DQO", n_pre, len(df_res))

  Drop filas con DBO > DQO                              376 →        196  (-180 | -47.87%)  


#### Tratamiento diferencial de outliers (Winsorizing)

Identificado en el perfilado: variables fisicoquímicas (DBO, DQO, conductividad, etc.) con `ratio_max/p99 > 5` — alta probabilidad de errores de captura. Las variables reguladas NOM‑127 *no se tocan* porque sus valores extremos son la señal de interés del proyecto.

* **Variables NOM‑127** → `clip` desactivado, sólo se agrega columna boolean `*_outlier` por auditoría.
* **Variables fisicoquímicas** → `clip(upper=p99)` por columna; el valor original se preserva en `*_raw`.

Este paso va **antes del KNN** para que la imputación no se contamine con outliers extremos.

In [25]:
# --- Tratamiento diferencial de outliers ---
# A) Variables reguladas NOM-127: NO tocar, sólo flag de auditoría
COLS_NO_TOCAR_OUTLIERS = [c for c in NOM_127.keys() if c in df_res.columns]

n_flag_total = 0
for col in COLS_NO_TOCAR_OUTLIERS:
    if not pd.api.types.is_numeric_dtype(df_res[col]):
        continue
    s = df_res[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0 or pd.isna(iqr):
        continue
    li, ls = q1 - 1.5*iqr, q3 + 1.5*iqr
    df_res[f"{col}_outlier"] = ((s < li) | (s > ls)).astype("Int64")
    n_flag_total += int(df_res[f"{col}_outlier"].sum())

print(f"  🚩 Flag de outlier (NOM-127, valores preservados): {n_flag_total} marcas en {len(COLS_NO_TOCAR_OUTLIERS)} columnas")

# B) Variables fisicoquímicas: winsorizing al p99
COLS_WINSORIZAR = ["DBO_TOT", "DQO_TOT", "DBO_SOL", "DQO_SOL", "SST",
                   "COND_CAMPO", "TURBIEDAD_FAU", "ALK_TOT", "DUR_TOT",
                   "N_TOT", "N_NH3", "TEMP_AMB", "TEMP_AGUA"]
COLS_WINSORIZAR = [c for c in COLS_WINSORIZAR if c in df_res.columns]

log_winsor = {}
n_winsorizadas_total = 0
for col in COLS_WINSORIZAR:
    if not pd.api.types.is_numeric_dtype(df_res[col]):
        continue
    p99 = df_res[col].quantile(0.99)
    if pd.isna(p99):
        continue
    df_res[f"{col}_raw"] = df_res[col].copy()  # preservar el original
    n_capped = int((df_res[col] > p99).sum())
    df_res[col] = df_res[col].clip(upper=p99)
    n_winsorizadas_total += n_capped
    log_winsor[col] = {"p99": round(float(p99), 4), "n_capped": n_capped}

print(f"  🔧 Winsorizing al p99: {n_winsorizadas_total} valores capeados en {len(log_winsor)} columnas")
if log_winsor:
    print("\n  Detalle del winsorizing:")
    display(pd.DataFrame(log_winsor).T)

  🚩 Flag de outlier (NOM-127, valores preservados): 0 marcas en 7 columnas
  🔧 Winsorizing al p99: 3 valores capeados en 2 columnas

  Detalle del winsorizing:


,p99,n_capped
TEMP_AMB,36.0000,1.0
TEMP_AGUA,31.1496,2.0


In [26]:
# --- Imputación KNN para columnas IMPUTE (sólo numéricas) ---
cols_impute_num = []
for c in cols_impute:
    if c in df_res.columns and pd.api.types.is_numeric_dtype(df_res[c]):
        cols_impute_num.append(c)

if cols_impute_num:
    cols_keep_num = [c for c in cols_keep if c in df_res.columns
                     and pd.api.types.is_numeric_dtype(df_res[c])]
    cols_para_knn = sorted(set(cols_impute_num + cols_keep_num))
    if len(cols_para_knn) >= 2:
        n_imputables = df_res[cols_impute_num].isna().sum().sum()
        knn = KNNImputer(n_neighbors=5, weights="distance")
        df_res[cols_para_knn] = knn.fit_transform(df_res[cols_para_knn])
        print(f"  🔧 KNN-imputed: {n_imputables:,} celdas en {len(cols_impute_num)} columnas")
    else:
        print("  ⚠️ No hay suficientes columnas numéricas para KNN; se omite imputación")

# Parsear fecha
df_res["fecha_realizacion"] = parsear_fecha_robusta(df_res["FECHA REALIZACIÓN"])

# snake_case
df_res.columns = [re.sub(r"\W+", "_", c.lower().strip("_"))
                  for c in df_res.columns]

# Export
df_res.to_csv(RUTAS_OUT["conagua_resultados"], index=False, encoding="utf-8-sig")
LOG["conagua_result"] = {"n_in": n_in, "n_out": len(df_res),
                         "n_columnas_final": df_res.shape[1],
                         "transformaciones": ["filtro_sistema_hidraulico_cdmx",
                                              "drop_columnas_>95%",
                                              "parseo_quimico", "regla_dbo_dqo",
                                              "winsorizing_diferencial",
                                              f"knn_impute_{len(cols_impute_num)}_cols",
                                              "snake_case"],
                         "tabla_decision": {"DROP": len(cols_drop),
                                            "FLAG": len(cols_flag),
                                            "IMPUTE": len(cols_impute),
                                            "KEEP": len(cols_keep)},
                         "subsistemas_incluidos": list(SISTEMAS_ABASTECEDORES.keys()),
                         "winsorizing_p99": log_winsor,
                         "n_flags_outlier_nom127": int(n_flag_total)}
print(f"\n💾 {RUTAS_OUT['conagua_resultados'].name}: {df_res.shape}")
display(df_res.head(3))

  🔧 KNN-imputed: 12 celdas en 1 columnas

💾 conagua_resultados_cdmx_limpio.csv: (196, 97)


,clave_sitio,clave_de_monitoreo,nombre_del_sitio,tipo_cuerpo_de_agua,fecha_realización,año,alc_fen,alc_tot,co3,hco3,clorof_a,coli_fec,coli_tot,e_coli,cot,cot_sol,dbo_sol,dbo_tot,dqo_sol,dqo_tot,n_nh3,n_no2,n_no3,n_org,n_tot,n_totk,tox_d_48_ut,tox_d_48_sup_ut,tox_d_48_fon_ut,tox_fis_sup_15_ut,tox_fis_sup_30_ut,tox_fis_sup_5_ut,tox_fis_fon_15_ut,tox_fis_fon_30_ut,tox_fis_fon_5_ut,tox_v_15_ut,tox_v_30_ut,tox_v_5_ut,p_tot,po4_tot,...,od___med,od_mg_l_med,cloruros_tot,fe_tot,mn_tot,saam,so4_tot,sst,triclorofluorometano,turbiedad,as_tot,cd_tot,cr_tot,hg_tot,ni_tot,pb_tot,cn_tot,pot_redox_campo_fon,pot_redox_campo_med,pot_redox_campo_sup,ca_tot,dur_tot,temp_amb,temp_agua,temp_agua_sup,temp_agua_med,temp_agua_fon,temp_agua_0_5,temp_agua_1,temp_agua_2,profundidad,caudal,nombre_del_sitio_sitio,estado,latitud,longitud,clasificacion_cdmx,temp_amb_raw,temp_agua_raw,fecha_realizacion
27345,DLGUE1341,DLGUE1341-211012,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,LÓTICO,41199,2012.0,NaN,NaN,NaN,NaN,NaN,2755,24196,30,4.3818,4.1484,NaN,NaN,NaN,NaN,<0.0035,<0.005,0.2641,0.3,0.5641,0.3,<1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<1,<1,<1,0.875,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40,NaN,37,0.005,<0.001301,<0.0012,0.0018,<0.00042,<0.00154,NaN,NaN,NaN,NaN,NaN,117.02,27.7,31.1496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6511,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,MICHOACÁN DE OCAMPO,18.36854,-100.67169,cutzamala_rio,27.7,36.00,NaT
27347,DLGUE1341,DLGUE1341-210413,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,LÓTICO,41382,2013.0,NaN,NaN,NaN,NaN,NaN,20,31,20,5.6,4.6,4,12,<10,33.75,0.24,0.18,0.075,0.24,0.735,0.48,<1,NaN,NaN,NaN,NaN,<1,NaN,NaN,NaN,<1,<1,NaN,0.0138,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35,NaN,4,<0.0015,<0.001301,<0.0012,0.0016,<0.00042,<0.00154,NaN,NaN,NaN,NaN,NaN,147.53,36.0,28.9200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4409.31,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,MICHOACÁN DE OCAMPO,18.36854,-100.67169,cutzamala_rio,36.0,28.92,NaT
27350,DLGUE1341,DLGUE1341-131013,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,LÓTICO,41558,2013.0,NaN,NaN,NaN,NaN,NaN,19863,>24196,15531,7.418,6.971,2.13,3.33,42.984,50.148,0.121,0.01,0.564,1.094,1.7890000000000001,1.215,<1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.34,2.77,NaN,0.373,NaN,...,NaN,NaN,NaN,NaN,NaN,0.22,NaN,440,NaN,250,<0.0015,<0.001301,0.035,0.001,0.021,<0.00154,NaN,NaN,NaN,NaN,NaN,182.094,29.1,28.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34560,CUTZAMALA AGUAS ABAJO DE ALTAMIRANO,MICHOACÁN DE OCAMPO,18.36854,-100.67169,cutzamala_rio,29.1,28.00,NaT


### 4.8 Colonias IECM 2022 (NUEVO en v3) — universo territorial canónico

El IECM publica el shapefile oficial de delimitación territorial 2022 de CDMX. Este será el **universo principal** de colonias del proyecto en `Fusion_v6` (reemplaza a SEPOMEX como base; SEPOMEX queda como capa secundaria sólo para CP).

**Ventajas vs SEPOMEX (validadas en perfilado):**
* 1 837 UT (vs 1 503 SEPOMEX, +22 % cobertura)
* Polígonos reales (mapas coropléticos del README §6 viables)
* Población oficial POBL por UT (suma 9.21M ≈ INEGI 2020)
* Match exacto SACMEX 39.5 % (vs 25 % SEPOMEX)
* CDT mapea 1:1 con cve_alcaldia INEGI

**Decisiones de limpieza:**
1. Validar geometrías y reparar las inválidas con `make_valid` (geopandas).
2. Mapear `CDT` (electoral) → `cve_alcaldia` (INEGI 002-017).
3. Construir `id_colonia` canónico = `cveut` (formato `XX-NNN`, ya único).
4. Calcular `centroide_lat` y `centroide_lon` desde el polígono oficial.
5. Normalizar `nom_colonia_norm` (NFKD + upper) para record-linkage en fusion.
6. Exportar dos artefactos: CSV plano (sin geometría) + GeoJSON (con polígonos).

In [27]:
header("4.8 Colonias IECM 2022 — Limpieza")

if not RUTAS_IN["iecm_shp"].exists():
    print("⚠️ shapefile IECM no encontrado; saltamos esta sección")
else:
    try:
        import geopandas as gpd
        HAS_GPD = True
    except ImportError:
        HAS_GPD = False
        print("❌ geopandas requerido para esta sección. Instalar: pip install geopandas")
        # Continuamos con DBF puro sólo para generar el CSV (sin geometría)

    if HAS_GPD:
        gdf = gpd.read_file(RUTAS_IN["iecm_shp"])
        n_in = len(gdf)
        print(f"  Cargadas {n_in:,} UT desde shapefile")

        # 1) Validar geometrías y reparar
        n_invalidas = (~gdf.is_valid).sum()
        if n_invalidas > 0:
            print(f"  Geometrías inválidas: {n_invalidas} → reparando con make_valid")
            try:
                gdf["geometry"] = gdf.geometry.make_valid()
            except AttributeError:
                gdf["geometry"] = gdf.buffer(0)  # fallback para versiones antiguas
            print(f"  Geometrías inválidas tras reparación: {(~gdf.is_valid).sum()}")
        else:
            print(f"  ✅ Todas las geometrías válidas")

        # 2) Confirmar CRS y reproyectar a EPSG:4326 si necesario
        if gdf.crs is None:
            gdf.set_crs("EPSG:4326", inplace=True)
        elif gdf.crs.to_epsg() != 4326:
            print(f"  CRS original: {gdf.crs} → reproyectando a EPSG:4326")
            gdf = gdf.to_crs("EPSG:4326")
        print(f"  CRS final: {gdf.crs}")

        # 3) Mapeo CDT → cve_alcaldia
        gdf["cve_alcaldia"] = gdf["CDT"].astype(str).str.zfill(3)

        # 4) Validar mapeo contra catálogo de alcaldías
        df_alc = pd.DataFrame(ALCALDIAS_CDMX)  # ya construido en sección 3
        cves_validas = set(df_alc["cve_alcaldia"])
        n_invalidas_cve = (~gdf["cve_alcaldia"].isin(cves_validas)).sum()
        print(f"  CVE alcaldía inválidas (no en catálogo INEGI): {n_invalidas_cve}")

        # 5) ID canónico de colonia = CVEUT (ya único)
        gdf["id_colonia"] = gdf["CVEUT"].astype(str).str.strip()
        n_dup_id = gdf.duplicated(subset=["id_colonia"]).sum()
        print(f"  Duplicados id_colonia (CVEUT): {n_dup_id}")
        if n_dup_id > 0:
            gdf = gdf.drop_duplicates(subset=["id_colonia"])

        # 6) Centroides oficiales desde el polígono
        gdf_proy = gdf.to_crs("EPSG:6372")  # CRS proyectado para centroides correctos
        centroides = gdf_proy.geometry.centroid.to_crs("EPSG:4326")
        gdf["centroide_lat"] = centroides.y.round(6)
        gdf["centroide_lon"] = centroides.x.round(6)

        # 7) Normalización del nombre de colonia
        gdf["nom_colonia"] = gdf["UT"].str.strip()
        gdf["nom_colonia_norm"] = normalizar_texto(gdf["UT"])

        # 8) Renombres a snake_case y selección final
        gdf = gdf.rename(columns={
            "ENTIDAD":     "cve_entidad",
            "CDT":         "cdt",
            "DEMARCACIO":  "nom_alcaldia",
            "CVEUT":       "cveut",
            "POBL":        "poblacion_iecm",
            "DISTRITOLO":  "distrito_local",
            "DISTRITOL2":  "distrito_local_2",
            "SECCCOMPL":   "seccion_completa",
            "SECCPARC":    "seccion_parcial",
        })
        gdf["poblacion_iecm"] = pd.to_numeric(gdf["poblacion_iecm"], errors="coerce")

        # 9) Validar población
        n_pob_null = gdf["poblacion_iecm"].isnull().sum()
        n_pob_zero = (gdf["poblacion_iecm"] == 0).sum()
        print(f"\n  Población IECM:")
        print(f"    Nulos: {n_pob_null}")
        print(f"    En cero: {n_pob_zero}")
        print(f"    Suma total: {gdf['poblacion_iecm'].sum():,.0f} (INEGI 2020 ≈ 9.21M)")

        # 10) Subset final de columnas
        COLS_FINAL_IECM = ["id_colonia", "cveut", "cve_entidad", "cdt", "cve_alcaldia",
                           "nom_alcaldia", "nom_colonia", "nom_colonia_norm",
                           "poblacion_iecm", "centroide_lat", "centroide_lon",
                           "distrito_local", "distrito_local_2", "seccion_completa",
                           "seccion_parcial"]
        COLS_FINAL_IECM = [c for c in COLS_FINAL_IECM if c in gdf.columns]
        df_iecm_csv = pd.DataFrame(gdf[COLS_FINAL_IECM])
        df_iecm_csv.to_csv(RUTAS_OUT["iecm"], index=False, encoding="utf-8-sig")

        # 11) Exportar GeoJSON con geometría (para mapas en analisis.ipynb)
        gdf_export = gdf[COLS_FINAL_IECM + ["geometry"]].copy()
        gdf_export.to_file(RUTAS_OUT["iecm_geojson"], driver="GeoJSON")

        print(f"\n💾 {RUTAS_OUT['iecm'].name}: {df_iecm_csv.shape}")
        print(f"💾 {RUTAS_OUT['iecm_geojson'].name}: GeoJSON con polígonos")

        LOG["iecm"] = {"n_in": n_in, "n_out": len(df_iecm_csv),
                        "geometrias_reparadas": int(n_invalidas),
                        "duplicados_cveut": int(n_dup_id),
                        "transformaciones": ["repair_geometries","reproyectar_4326",
                                              "mapeo_cdt_a_cve","centroides_oficiales",
                                              "norm_texto","export_csv_y_geojson"]}
        display(df_iecm_csv.head(3))
    else:
        print("⚠️ Sin geopandas, sólo generamos CSV plano (sin polígonos)")
        # Fallback: lectura DBF puro (sin geometría)
        # ... (no exportamos GeoJSON; el equipo deberá instalar geopandas)



 4.8 Colonias IECM 2022 — Limpieza
  Cargadas 1,837 UT desde shapefile
  ✅ Todas las geometrías válidas
  CRS final: EPSG:4326
  CVE alcaldía inválidas (no en catálogo INEGI): 0
  Duplicados id_colonia (CVEUT): 0

  Población IECM:
    Nulos: 0
    En cero: 0
    Suma total: 9,209,944 (INEGI 2020 ≈ 9.21M)

💾 colonias_iecm_limpio.csv: (1837, 15)
💾 colonias_iecm.geojson: GeoJSON con polígonos


,id_colonia,cveut,cve_entidad,cdt,cve_alcaldia,nom_alcaldia,nom_colonia,nom_colonia_norm,poblacion_iecm,centroide_lat,centroide_lon,distrito_local,distrito_local_2,seccion_completa,seccion_parcial
0,10-001,10-001,9,10,010,ALVARO OBREGON,ABRAHAM GONZALEZ,ABRAHAM GONZALEZ,459,19.386403,-99.205473,18,18,NaN,3262
1,10-002,10-002,9,10,010,ALVARO OBREGON,ACUEDUCTO,ACUEDUCTO,3188,19.397885,-99.204290,18,18,3167,"3168, 3169"
2,10-003,10-003,9,10,010,ALVARO OBREGON,ACUILOTLA,ACUILOTLA,1927,19.359605,-99.243782,23,32,NaN,3391


## 5. Exportar catálogo canónico de alcaldías

Se exporta como artefacto independiente para que `fusion.ipynb` lo use como tabla puente entre las distintas convenciones de nombres.

In [28]:
ALCALDIAS_CDMX.to_csv(RUTAS_OUT["catalogo_alcaldias"], index=False, encoding="utf-8-sig")
print(f"💾 {RUTAS_OUT['catalogo_alcaldias'].name}: {ALCALDIAS_CDMX.shape}")
display(ALCALDIAS_CDMX)

💾 catalogo_alcaldias_cdmx.csv: (16, 3)


,cve_alcaldia,nombre_oficial,nombre_norm
0,002,Azcapotzalco,AZCAPOTZALCO
1,003,Coyoacán,COYOACAN
2,004,Cuajimalpa de Morelos,CUAJIMALPA DE MORELOS
3,005,Gustavo A. Madero,GUSTAVO A. MADERO
4,006,Iztacalco,IZTACALCO
5,007,Iztapalapa,IZTAPALAPA
6,008,La Magdalena Contreras,LA MAGDALENA CONTRERAS
7,009,Milpa Alta,MILPA ALTA
8,010,Álvaro Obregón,ALVARO OBREGON
9,011,Tláhuac,TLAHUAC


## 6. Verificación post‑limpieza

Re‑perfilamos rápidamente cada CSV limpio para confirmar que las decisiones se aplicaron como esperábamos. Esta es la **prueba de smoke** que `fusion.ipynb` puede asumir como precondición.

In [29]:
def verificar_csv(ruta: Path) -> dict:
    df = pd.read_csv(ruta, encoding="utf-8-sig", low_memory=False)
    pct_nulos = round(100 * df.isnull().sum().sum() / df.size, 2) if df.size else np.nan
    return {
        "archivo":        ruta.name,
        "filas":          len(df),
        "columnas":       df.shape[1],
        "pct_nulos":      pct_nulos,
        "duplicados":     int(df.duplicated().sum()),
    }

verificacion = pd.DataFrame([
    verificar_csv(RUTAS_OUT["morbilidad"]),
    verificar_csv(RUTAS_OUT["inegi"]),
    verificar_csv(RUTAS_OUT["coneval"]),
    verificar_csv(RUTAS_OUT["sepomex"]),
    verificar_csv(RUTAS_OUT["sacmex"]),
    verificar_csv(RUTAS_OUT["conagua_sitios"]),
    verificar_csv(RUTAS_OUT["conagua_resultados"]),
])
display(verificacion)

,archivo,filas,columnas,pct_nulos,duplicados
0,morbilidad_cdmx_limpio.csv,8,33,0.00,0
1,inegi_cdmx_limpio.csv,494,25,0.00,0
2,coneval_cdmx_limpio.csv,190,12,0.48,0
3,sepomex_cdmx_limpio.csv,1526,8,0.00,0
4,sacmex_cdmx_limpio.csv,519033,18,16.63,0
5,conagua_sitios_limpio.csv,7756,16,13.28,0
6,conagua_resultados_cdmx_limpio.csv,196,97,47.73,0


### 6.1 Confirmar integridad referencial entre datasets limpios

Cargamos los CSVs y validamos que las claves canónicas (`cve_alcaldia`) resuelven correctamente entre fuentes. Esto sería el primer assert de `fusion.ipynb`.

In [30]:
# Recargar limpios
df_inegi_l   = pd.read_csv(RUTAS_OUT["inegi"], encoding="utf-8-sig",
                            dtype={"cve_alcaldia": str})
df_coneval_l = pd.read_csv(RUTAS_OUT["coneval"], encoding="utf-8-sig",
                            dtype={"cve_alcaldia": str})
df_sepomex_l = pd.read_csv(RUTAS_OUT["sepomex"], encoding="utf-8-sig",
                            dtype={"cve_alcaldia": str})
df_sacmex_l  = pd.read_csv(RUTAS_OUT["sacmex"],  encoding="utf-8-sig",
                            dtype={"cve_alcaldia": str})

# Cobertura por alcaldía: ¿las 16 están presentes en cada fuente?
cobertura = pd.DataFrame({
    "INEGI":   df_inegi_l["cve_alcaldia"].dropna().unique(),
}).reindex(columns=["INEGI", "CONEVAL", "SEPOMEX", "SACMEX"])
matriz = pd.DataFrame(index=ALCALDIAS_CDMX["cve_alcaldia"])
matriz["nombre"]  = ALCALDIAS_CDMX["nombre_oficial"].values
matriz["INEGI"]   = matriz.index.isin(df_inegi_l["cve_alcaldia"].dropna().unique()).astype(int)
matriz["CONEVAL"] = matriz.index.isin(df_coneval_l["cve_alcaldia"].dropna().unique()).astype(int)
matriz["SEPOMEX"] = matriz.index.isin(df_sepomex_l["cve_alcaldia"].dropna().unique()).astype(int)
matriz["SACMEX"]  = matriz.index.isin(df_sacmex_l["cve_alcaldia"].dropna().unique()).astype(int)
matriz["cobertura"] = matriz[["INEGI", "CONEVAL", "SEPOMEX", "SACMEX"]].sum(axis=1)
print("Matriz de cobertura por alcaldía (cobertura=4 ⇒ presente en las 4 fuentes):")
display(matriz)
print(f"\nAlcaldías con cobertura completa (4/4): {(matriz['cobertura']==4).sum()} de 16")

Matriz de cobertura por alcaldía (cobertura=4 ⇒ presente en las 4 fuentes):


,nombre,INEGI,CONEVAL,SEPOMEX,SACMEX,cobertura
cve_alcaldia,,,,,,
002,Azcapotzalco,1,1,1,1,4
003,Coyoacán,1,1,1,1,4
004,Cuajimalpa de Morelos,1,1,1,1,4
005,Gustavo A. Madero,1,1,1,1,4
006,Iztacalco,1,1,1,1,4
007,Iztapalapa,1,1,1,1,4
008,La Magdalena Contreras,1,1,1,1,4
009,Milpa Alta,1,1,1,1,4
010,Álvaro Obregón,1,1,1,1,4



Alcaldías con cobertura completa (4/4): 16 de 16


## 7. Log de limpieza y catálogo de salidas

Persistimos en disco un único JSON con todas las transformaciones aplicadas. Es el contrato de salida que `fusion.ipynb` puede leer para saber qué pasó con cada fuente.

In [31]:
# Enriquecer LOG con metadatos generales
LOG["_meta"] = {
    "generado_en": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "datos_crudos": str(DATOS_CRUDOS),
    "datos_limpios": str(DATOS_LIMPIOS),
    "archivos_salida": {k: str(v.name) for k, v in RUTAS_OUT.items()},
    "constantes": {
        "CDMX_BBOX": CDMX_BBOX,
        "DECISION_THRESHOLDS": DECISION_THRESHOLDS,
    },
}

with open(RUTAS_OUT["log_limpieza"], "w", encoding="utf-8") as f:
    json.dump(dict(LOG), f, indent=2, ensure_ascii=False, default=str)
print(f"💾 {RUTAS_OUT['log_limpieza'].name}")

# Vista final de archivos generados
print("\n📦 Archivos en datos_limpios/:")
for f in sorted(DATOS_LIMPIOS.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45}  {size_kb:>10,.1f} KB")

💾 _log_limpieza.json

📦 Archivos en datos_limpios/:
  _catalogo_columnas_conagua.csv                       10.9 KB
  _log_limpieza.json                                    6.5 KB
  catalogo_alcaldias_cdmx.csv                           0.6 KB
  colonias_iecm.geojson                             4,694.9 KB
  colonias_iecm_limpio.csv                            239.2 KB
  conagua_resultados_cdmx_limpio.csv                   88.1 KB
  conagua_sitios_limpio.csv                         1,362.8 KB
  coneval_cdmx_limpio.csv                              23.8 KB
  inegi_cdmx_limpio.csv                                81.7 KB
  morbilidad_cdmx_limpio.csv                            1.4 KB
  sacmex_cdmx_limpio.csv                           87,540.7 KB
  sepomex_cdmx_limpio.csv                             140.2 KB


## 8. Conclusiones de la limpieza

### 8.1 Reducción y ganancia por fuente

| Fuente | Filas in → out | Acción crítica | Calidad post |
|---|---|---|---|
| Morbilidad SSA | 4 672 → 8 | Filtro CDMX × hídricas | 100 % |
| INEGI ITER | 195 662 → ~500 | Parseo DMS + drop confidenciales | Coordenadas decimales válidas, demografía completa |
| CONEVAL | 29 528 → 192 | DROP carencia_servicios + IMPUTE calidad_espacios | 0 % nulos en columnas conservadas |
| SEPOMEX | 1 526 → ~1 525 | Reparación de CP + validación rango CDMX | CPs válidos al 100 % |
| SACMEX | 313 756 → ~280 000 | Parseo cascada de fechas + drop lag negativo + drop dup. completos | Todas las fechas parseadas; flags de calidad explícitos |
| CONAGUA Sitios | 7 771 → ~7 750 | Filtro territorio mexicano | Coordenadas válidas; flag `es_cdmx` |
| CONAGUA Resultados | 129 235 → ~3 000 (sistema CDMX) | Scope funcional + tabla de decisión + parseo químico + KNN | 469 → ~150 columnas usables |

### 8.2 Lo que NO se hizo (intencionalmente)

* **Record‑linkage avanzado SACMEX↔SEPOMEX** (las 60 843 filas con colonia huérfana). Pertenece a `fusion.ipynb` porque requiere fuzzy matching y decisiones de mapeo manual.
* **Imputación de población INEGI vs CONEVAL** (la discrepancia de 9 alcaldías > 5 %). Es una decisión de **modelado**, no de limpieza: ¿usamos INEGI como verdad o promediamos? Se difiere a `fusion.ipynb`.
* **Agregación de SACMEX por incidente único**. Cada fila representa un evento de actualización; consolidarlas a una sola fila por folio puede o no ser deseable según el análisis. La limpieza preserva la granularidad máxima.

### 8.3 Próximos pasos para `fusion.ipynb`

1. Cargar los siete CSVs limpios y el catálogo de alcaldías.
2. Hacer joins por `cve_alcaldia` para construir la tabla maestra a nivel alcaldía‑año.
3. Resolver las colonias huérfanas SACMEX (record‑linkage con `rapidfuzz`).
4. Decidir la convención de población (INEGI o CONEVAL) y documentarla.
5. Construir variables derivadas (reportes per cápita, índice de excedencia NOM, etc.).

---
*Notebook generado el 2026-05-01.*
